# Matplotlib Feature Tracking Algorithm -- Precipitation

### Import Packages

In [ ]:
# Import necessary libraries for identifying contours and plotting
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from skimage.measure import label, regionprops
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import cartopy.crs as ccrs
from matplotlib.path import Path

Load in the data using `xarray`, and extract the variable of interest.

We want to look at the total precipitation of the extra-tropical cyclones, which is stored under the `ETC_tp` variable.

In [ ]:
# Load in precipitation dataset
ds = xr.open_dataset('data/precip_ex.nc')

# Extra-tropical cyclone total precipitation
da = ds["ETC_tp"]

We will find the contours using `Matplotlib`. 

First, we initialize the contour-detection parameters, and extract the latitude and longitude coordinates from the dataset. We then convert the coordinates into 2D grids so the found contours can be plotted in geographic space.

In [ ]:
# Define parameters for contour finding
threshold = 100000
min_area = 20
min_overlap = 0.2  

# Extract latitutde and longitude coordinates
lat = da[da.dims[1]].values # assumes dims like (time, lat, lon)
lon = da[da.dims[2]].values

# Make 2D grids for contour
lon2d, lat2d = np.meshgrid(lon, lat)

We define a function that detects connected regions with total precipitation levels below the threshold.

In [ ]:
def track_precip_features_matplotlib(
    da,
    threshold=threshold,
    min_area=min_area,
    min_overlap=min_overlap
):
    """
    Track regions with low total precipitation (total precipitation <= threshold) using Matplotlib contours.
    """
    ny, nx = da.isel(time=0).shape
    yy, xx = np.mgrid[0:ny, 0:nx]
    points = np.column_stack([xx.ravel(), yy.ravel()])

    tracked = []
    prev = {}
    next_track_id = 1

    for t in range(da.sizes["time"]):
        field = da.isel(time=t).values
        fig, ax = plt.subplots()
        cs = ax.contour(field, levels=[threshold])
        plt.close(fig)

        current = {}
        frame_info = []

        for seg in cs.allsegs[0]:
            path = Path(seg)
            mask = path.contains_points(points).reshape(ny, nx)

            # keep low precipitation regions only
            mask = mask & (field <= threshold)
            if mask.sum() < min_area:
                continue

            y, x = np.where(mask)
            centroid = (y.mean(), x.mean())

            best_id = None
            best_overlap = 0

            for track_id, old_mask in prev.items():
                overlap = np.logical_and(mask, old_mask).sum() / mask.sum()

                if overlap > best_overlap:
                    best_overlap = overlap
                    best_id = track_id

            if best_overlap < min_overlap:
                best_id = next_track_id
                next_track_id += 1 

            current[best_id] = mask

            frame_info.append({
                "track_id": best_id,
                "mask": mask,
                "centroid_rc": centroid,
            })

        tracked.append(frame_info)
        prev = current

    return tracked

Run the tracking function for each time step, and save the results in a variable `tracked`.

In [ ]:
tracked = track_precip_features_matplotlib(
    da,
    threshold=threshold,
    min_area=min_area,
    min_overlap=min_overlap,
)

In the visualization, we will have a colorbar that represent precipitation levels. We want the bar to accurately depict the bulk of the data, so we limit the lower end to the 1st percentile of the data, and the upper end to the 99th percentile.

In [ ]:
# Find 1st percentile and 99th percentile of total precipitation values
vals = da.values
vmin = np.nanpercentile(vals, 1)
vmax = np.nanpercentile(vals, 99)

levels = np.linspace(vmin, vmax, 16) # evenly spaced intervals
norm = mcolors.Normalize(vmin=vmin, vmax=vmax) # normalize
cmap = 'turbo' # colormap

We can animate the storm tracked over time by plotting the low precipitation regions at each timestep.

In [ ]:
def update(t):
    """
    Animation function, goes through each timestep and its tracked precipitation.
    """
    ax.clear()
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    field = da.isel(time=t).values # total precipitation data for current timestep

    # Colored contour lines
    ax.contour(
        lon2d, lat2d, field,
        levels=levels,
        cmap=cmap,
        norm=norm,
        linewidths=0.9,
        transform=ccrs.PlateCarree()
    )

    # Tracked feature outlines
    for feat in tracked[t]:
        mask = feat["mask"]

        ax.contour(
            lon2d, lat2d, mask.astype(int),
            levels=[0.5],
            colors="black",
            linewidths=1.6,
            transform=ccrs.PlateCarree()
        )

        y, x = feat["centroid_rc"]
        iy = int(round(y))
        ix = int(round(x))

        ax.plot(
            lon[ix], lat[iy],
            "ko", ms=3,
            transform=ccrs.PlateCarree()
        )

        ax.text(
            lon[ix], lat[iy],
            str(feat["track_id"]),
            fontsize=8,
            color="black",
            ha="left",
            va="bottom",
            transform=ccrs.PlateCarree(),
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1)
        )

    # Create outlines for continents/coastlines and gridlines
    ax.coastlines(color="0.55", linewidth=0.6)
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(f"Matplotlib contour-tracked precipitation features | time index = {t}")

In [ ]:
# Create map
fig, ax = plt.subplots(figsize=(12, 4), subplot_kw={"projection": ccrs.PlateCarree()})
extent = [lon.min(), lon.max(), lat.min(), lat.max()] # geographic boundaries of the map
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Create colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=ax,
    orientation="horizontal",
    pad=0.15
)
cbar.set_label("ETC_p")

# Create animation
anim = FuncAnimation(
    fig,
    update,
    frames=da.sizes["time"],
    interval=300,
    blit=False,
)

plt.close(fig)
HTML(anim.to_jshtml())